[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-biopython-course/blob/main/notebooks/05_ncbi_entrez.ipynb)

# 05 · Searching and Downloading from NCBI with Entrez

*Practical Biopython Course · Hamza Hentabli, Ph.D. · ~50 min*

NCBI hosts GenBank, RefSeq, PubMed, and dozens of other databases. `Bio.Entrez` wraps the
**E-utilities** web API. This notebook queries NCBI live and, if the network is unavailable (or you set
`COURSE_OFFLINE=1`), falls back to cached copies of the same queries in `data/entrez_cache/`, so it
always runs.

**Learning objectives.** By the end of this notebook you will be able to:

- explain the Entrez E-utilities (`esearch`, `esummary`, `efetch`) and NCBI's usage rules
- search NCBI Nucleotide and PubMed from Python and parse the results
- download a GenBank record and save it locally
- write network code that is polite (email, rate limits) and robust (timeouts, offline cache)

In [1]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Install requirements.txt once and this cell does nothing extra.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'biopython'], check=True)
    if not os.path.isdir("practical-biopython-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-biopython-course.git"], check=True)
    DATA_DIR = "practical-biopython-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR)

Running on Colab: False | data folder: ../data


## 1. Rules of the road

NCBI asks every user to:

1. **Identify yourself** with `Entrez.email` (they contact you before blocking you if something goes wrong).
2. Make **no more than 3 requests per second** (10 with a free API key).
3. Run large jobs on weekends or between 9 pm and 5 am US Eastern time.

Replace the placeholder below with your own email address.

In [2]:
import os, time
from io import StringIO
from Bio import Entrez, SeqIO
import pandas as pd

Entrez.email = "your.name@university.edu"   # <-- put your own email here
Entrez.tool = "practical-biopython-course"
# Entrez.api_key = "..."                    # optional: raises the limit to 10 requests/second

CACHE = f"{DATA_DIR}/entrez_cache"
OFFLINE = os.environ.get("COURSE_OFFLINE") == "1"

## 2. A polite, robust helper

Every call goes through one function that waits between requests, retries on failure, and falls back
to the cache. Writing this once saves you from flaky scripts later.

In [3]:
_last_call = [0.0]

def entrez_raw(func, cache_file, retries=3, **kwargs):
    """Call an Entrez function and return the raw response (bytes/str).
    Falls back to data/entrez_cache/<cache_file> when NCBI cannot be reached."""
    if not OFFLINE:
        for attempt in range(retries):
            try:
                wait = 0.34 - (time.time() - _last_call[0])      # <= 3 requests per second
                if wait > 0:
                    time.sleep(wait)
                _last_call[0] = time.time()
                with func(**kwargs) as handle:
                    data = handle.read()
                print(f"live NCBI response ({func.__name__})")
                return data
            except Exception as err:                            # network errors, HTTP 429/5xx, timeouts
                print(f"attempt {attempt + 1} failed: {err}")
                time.sleep(2 ** attempt)
    print(f"using cached copy: {cache_file}")
    mode = "rb" if cache_file.endswith(".xml") else "r"
    with open(f"{CACHE}/{cache_file}", mode) as fh:
        return fh.read()

def as_handle(data):
    from io import BytesIO
    return BytesIO(data) if isinstance(data, bytes) else StringIO(data)

## 3. `esearch`: find record IDs

The search syntax is the same as on the NCBI website. Field tags such as `[Gene Name]` and `[Organism]`
make queries precise; the `refseq_select` filter keeps one representative transcript per gene.

In [4]:
term = "EGFR[Gene Name] AND Homo sapiens[Organism] AND refseq_select[filter]"
raw = entrez_raw(Entrez.esearch, "esearch_nucleotide_EGFR.xml", db="nucleotide", term=term, retmax=20)
result = Entrez.read(as_handle(raw))
print("hits:", result["Count"], "| IDs:", result["IdList"])
print("NCBI interpreted the query as:", result["QueryTranslation"])

live NCBI response (esearch)
hits: 1 | IDs: ['1519245592']
NCBI interpreted the query as: EGFR[Gene Name] AND "Homo sapiens"[Organism] AND refseq_select[filter]


## 4. `esummary`: a quick look before downloading

In [5]:
raw = entrez_raw(Entrez.esummary, "esummary_NM_005228.xml", db="nucleotide", id="NM_005228")
summary = Entrez.read(as_handle(raw))[0]
for key in ["Caption", "Title", "Length", "UpdateDate", "TaxId"]:
    print(f"{key:>10}: {summary[key]}")

live NCBI response (esummary)
   Caption: NM_005228
     Title: Homo sapiens epidermal growth factor receptor (EGFR), transcript variant 1, mRNA
    Length: IntegerElement(9905, attributes={})
UpdateDate: 2026/04/13
     TaxId: IntegerElement(9606, attributes={})


## 5. `efetch`: download the record

`rettype="gb"` returns GenBank flat-file text, `rettype="fasta"` returns FASTA. Save what you download:
your analysis must be reproducible even if the record is later updated.

In [6]:
raw = entrez_raw(Entrez.efetch, "NM_005228.gb", db="nucleotide", id="NM_005228", rettype="gb", retmode="text")
record = SeqIO.read(as_handle(raw), "genbank")
print(record.id, "|", len(record), "nt |", record.annotations["date"])

os.makedirs("output", exist_ok=True)
SeqIO.write(record, "output/NM_005228_downloaded.gb", "genbank")
print("saved to output/NM_005228_downloaded.gb")

live NCBI response (efetch)
NM_005228.5 | 9905 nt | 13-APR-2026
saved to output/NM_005228_downloaded.gb


The version suffix (`.5`) changes when the sequence changes. Always report accession **with** version.

## 6. Searching PubMed

Entrez is not only sequences. Here we look for review articles on resistance to osimertinib, the
third-generation EGFR inhibitor.

In [7]:
pterm = "osimertinib AND EGFR AND resistance AND review[pt]"
raw = entrez_raw(Entrez.esearch, "esearch_pubmed_osimertinib.xml", db="pubmed", term=pterm, retmax=10, sort="relevance")
pubmed = Entrez.read(as_handle(raw))
print("reviews found:", pubmed["Count"])
ids = pubmed["IdList"]

raw = entrez_raw(Entrez.esummary, "esummary_pubmed_osimertinib.xml", db="pubmed", id=",".join(ids))
papers = pd.DataFrame([{"PMID": d["Id"], "year": d["PubDate"][:4], "journal": d["Source"],
                        "title": d["Title"][:90]} for d in Entrez.read(as_handle(raw))])
papers

live NCBI response (esearch)
reviews found: 337


live NCBI response (esummary)


,PMID,year,journal,title
0,31564718,2019,Br J Cancer,Resistance mechanisms to osimertinib in EGFR-m...
1,35534623,2022,Nat Rev Clin Oncol,Third-generation EGFR and ALK inhibitors: mech...
2,40133478,2025,Lung,Update 2025: Management of Non‑Small-Cell Lung...
3,34558640,2021,Int J Oncol,Mechanisms and management of 3rd‑generation EG...
4,38572595,2024,Expert Opin Emerg Drugs,Non-small cell lung cancer: an update on emerg...
5,41219394,2026,Nat Rev Clin Oncol,Navigating the landscape of EGFR TKI resistanc...
6,29462255,2018,Ann Oncol,Osimertinib and other third-generation EGFR TK...
7,36482474,2022,J Hematol Oncol,Therapeutic strategies for EGFR-mutated non-sm...
8,37060646,2023,Cancer Treat Rev,Acquired resistance mechanisms to osimertinib:...
9,39366135,2024,Cancer Treat Rev,Advancements in fourth-generation EGFR TKIs in...


If you run this live, the count and list will differ from the cached copy: PubMed grows every day.
That is exactly why you record the query **and** the retrieval date (see `data/provenance.json`).

In [8]:
import json
with open(f"{DATA_DIR}/provenance.json") as fh:
    print(json.dumps(json.load(fh)["entrez"], indent=2))

{
  "retrieved": "2026-11-24",
  "nucleotide_term": "EGFR[Gene Name] AND Homo sapiens[Organism] AND refseq_select[filter]",
  "pubmed_term": "osimertinib AND EGFR AND resistance AND review[pt]",
  "records": [
    "NM_005228"
  ]
}


## 7. Downloading many records

For hundreds of IDs, do **not** loop over `efetch` one ID at a time: pass a comma-separated list (up to a
few hundred per call), or use `usehistory="y"` with `WebEnv`/`query_key` and fetch in batches.

In [9]:
# Pattern for large downloads (not executed here to keep NCBI traffic low):
example = """
search = Entrez.read(Entrez.esearch(db="protein", term=term, usehistory="y"))
for start in range(0, int(search["Count"]), 200):
    with Entrez.efetch(db="protein", rettype="fasta", retmode="text", retstart=start, retmax=200,
                       webenv=search["WebEnv"], query_key=search["QueryKey"]) as h:
        out.write(h.read())
    time.sleep(0.4)
"""
print(example)


search = Entrez.read(Entrez.esearch(db="protein", term=term, usehistory="y"))
for start in range(0, int(search["Count"]), 200):
    with Entrez.efetch(db="protein", rettype="fasta", retmode="text", retstart=start, retmax=200,
                       webenv=search["WebEnv"], query_key=search["QueryKey"]) as h:
        out.write(h.read())
    time.sleep(0.4)



### ✏️ Exercise 1

Use `entrez_raw` + `Entrez.read` to run `einfo` on the `pubmed` database (cache file: none, so wrap it in
`try/except` and print a message if you are offline). Print the database description and record count.

In [10]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
try:
    if OFFLINE:
        raise ConnectionError("offline mode")
    with Entrez.einfo(db="pubmed") as h:
        info = Entrez.read(h)["DbInfo"]
    info = info[0] if isinstance(info, list) else info      # newer Biopython returns a list
    print(info["Description"], "|", info["Count"], "records")
except Exception as err:
    print("einfo needs the network:", err)
```

</details>

### ✏️ Exercise 2

From the downloaded `record`, print the protein ID and the first 30 residues of the CDS translation
qualifier.

In [11]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
cds = next(f for f in record.features if f.type == "CDS")
print(cds.qualifiers["protein_id"][0], cds.qualifiers["translation"][0][:30])
```

</details>

### ✏️ Exercise 3

Add a column `recent` to `papers` that is `True` for reviews from 2024 or later, and count them.

In [12]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
papers["recent"] = papers["year"].astype(int) >= 2024
print(papers["recent"].sum(), "of", len(papers), "reviews are from 2024 or later")
```

</details>

## Key takeaways

- `esearch` → IDs, `esummary` → quick metadata, `efetch` → full records.
- Set `Entrez.email`, stay under 3 requests/s, batch large downloads.
- Wrap network calls with retries and a cache so your notebook still runs tomorrow.

**Next:** [06 · Pairwise alignment](06_pairwise_alignment.ipynb).